In [ ]:
import math
from collections import deque
#Using a scalar class that represents the node in each graph for any kind of mathematical equation
#The whole point of the autograd is that each sort of individual operation has its own unique way of taking the derivative
#For example: if a = b + c, then da/db = 1 or da/dc = 1. However, if a = b*c, then da/db = c and da/dc = b. Each of these operations
#Have a different way of computing their local derivatives. In a bigger more convoluted equation like a = b*c + d/f - c+b,
#The equation can be reconstrcuted by the correct order of operations and the chain rule can be applied starting from the local derivative
#all the way up to the desired result.


class Scalar:
    #Each scalar has a set of children its DIRECTLY derived from. NOT the indirect. The indirect children 
    #can be obtained via traversing backwards in the graph
    def __init__(self, digit, children:set = (), operation = ""):
        self.digit = digit
        self.children = children
        self.operation = operation
        self.back = lambda: None
        #By default, the gradient is taken of a final expression that will be specified
        self.gradient = 0
    
    #Building out the basic operations.
    def __add__(self, other:Scalar):
        res = Scalar(self.digit + other.digit, {self, other}, "add")
        def back():
            print(f"add back called, res.gradient={res.gradient}")
            #The local derivative of an added expression w.r.t 1 element (ex: d(a + b + c)/da = 1) is 1
            #res.gradient is the derivative of the FINAL output with respect to this current res
            self.gradient += res.gradient
            #Same logic applies for other
            other.gradient += res.gradient
        res.back = back
        return res
    
    def __mul__(self, other:Scalar):
        res = Scalar(self.digit * other.digit, {self, other}, "mul")
        def back():
            print(f"mul back called, res.gradient={res.gradient}")
            #Local derivative (d(ab)/da = b) and (d(ab)/db = a) 
            #Multiply local with global derivative
            self.gradient += other.digit * res.gradient
            other.gradient += self.digit * res.gradient
        res.back = back
        return res
    
    #Using true div since we want floats
    def __truediv__(self, other:Scalar):
        res = Scalar(self.digit / other.digit, {self, other}, "div")
        def back():
            #Local derivative (d(a/b)/da = 1/b) and (d(a/b)/db = -a*b^-2)
            #Multiply with global derivative
            self.gradient += res.gradient * (1/other.digit)
            other.gradient += self.digit * (-(other.digit**-2)) * res.gradient
        res.back = back
        return res
    
    def __sub__(self, other:Scalar):
        res = Scalar(self.digit - other.digit, {self, other}, "sub")
        def back():
            #Local derivative (d(a-b)/da = 1) and (d(a-b)/db = -1)
            #Multiply with global derivative
            self.gradient += res.gradient
            other.gradient += -res.gradient
        res.back = back
        return res

    def __neg__(self):
        res = Scalar(-self.digit, {self}, "neg")
        def back():
            #Local derivative (d(-a)/da = -1)
            #Multiply with global derivative
            self.gradient += -1 * res.gradient
        res.back = back
        return res

    #arctan
    def tanh(self):
        res = Scalar(math.tanh(self.digit), {self}, "tanh")
        def back():
            #Local derivative (d(tanh(a))/da = 1 - tanh^2(a)) 
            #Multiply with global derivative
            self.gradient += (1 - (math.tanh(self.digit))**2) * res.gradient
        res.back = back
        return res

    def exp(self):
        res = Scalar(math.exp(self.digit), {self}, "exp")
        def back():
            #Local derivative (d(e^a)/da = e^a)
            #Multiply with global derivative
            self.gradient += math.exp(self.digit) * res.gradient
        res.back = back
        return res

    def log(self):
        res = Scalar(math.log(self.digit), {self}, "log")
        def back():
            #Local derivative (d(ln(a))/da = 1/a)
            #Multiply with global derivative
            self.gradient += (1 / self.digit) * res.gradient
        res.back = back
        return res

    def relu(self):
        res = Scalar(max(0, self.digit), {self}, "relu")
        def back():
            #Local derivative (d(relu(a))/da = 1 if a > 0, else 0)
            #Multiply with global derivative
            self.gradient += (1 if self.digit > 0 else 0) * res.gradient
        res.back = back
        return res

    def sigmoid(self):
        sig = 1 / (1 + math.exp(-self.digit))
        res = Scalar(sig, {self}, "sigmoid")
        def back():
            #Local derivative (d(sigmoid(a))/da = sigmoid(a) * (1 - sigmoid(a)))
            #Multiply with global derivative
            self.gradient += (sig * (1 - sig)) * res.gradient
        res.back = back
        return res

    def pow(self, other:Scalar):
        res = Scalar(math.pow(self.digit, other.digit), {self, other}, "pow")
        def back():
            #Local derivative (d(a^b)/da = b * a^(b-1)) and (d(a^b)/db = a^b * ln(a))
            #Multiply with global derivative
            self.gradient += (other.digit * math.pow(self.digit, other.digit - 1)) * res.gradient
            other.gradient += (res.digit * math.log(self.digit)) * res.gradient
        res.back = back
        return res

    #Reverse operations handle cases where a raw number is on the left (e.g. 2 + scalar, 3 * scalar).
    #Python falls back to these when the left operand doesn't know how to handle a Scalar.
    def __radd__(self, other):
        return Scalar(other) + self

    def __rmul__(self, other):
        return Scalar(other) * self

    def __rsub__(self, other):
        return Scalar(other) - self

    def __rtruediv__(self, other):
        return Scalar(other) / self
    
    #This is to prevent calling backprop manually. Kahns topological sort flattens the graph into a list that can be traversed in order
    #That way back can be called in the correct order
    def topological_sort(root_node:Scalar):
        num_dependancies = {}
        
        def collect_nodes(node:Scalar):
            if node not in num_dependancies:
                num_dependancies[node] = 0
            for c in node.children:
                collect_nodes(c)
        
        def obtain_dependancies():
            for n in num_dependancies:
                for c in n.children:
                    num_dependancies[c] += 1

        final_res = []
        collect_nodes(root_node)
        obtain_dependancies()
        print(num_dependancies)

        q = deque()
        for n in num_dependancies:
            if num_dependancies[n] == 0:
                q.append(n)

        while len(q) > 0:
            qLen = len(q)
            for _ in range(qLen):
                node:Scalar = q.pop()
                final_res.append(node)
                for c in node.children:
                    num_dependancies[c] -= 1
                    if num_dependancies[c] == 0:
                        q.append(c)
        
        for n in final_res:
            n.back()
